# Technical Weblink Finder & Ranker

A two-step pipeline using GPT-4o-mini:
1. **Find** the best weblinks for a given technical topic
2. **Score** each link for relevance (1-10) using structured JSON output
3. **Explain** the topic and surface the top-ranked links

Built as a Week 1 exercise for the LLM Engineering course.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
system_prompt = """
You are a helpful assistant that finds best weblinks 
for a given technical topic. You will be given a question 
and you need to find the best weblink that answers the question. 
You should only return a set of "\n" separated weblinks. If you cannot find a weblink, return 'No weblink found'.
You should not return any explanation text other than the weblink(s).
"""

In [ ]:
def get_user_prompt(question):
    return f"Question: {question}\nAnswer with weblink(s):"

In [ ]:
def get_weblinks(topic):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(topic)}
        ]
    )
    if response.choices[0].message.content.strip().lower() == 'no weblink found':
        print("No weblink found for the topic.")
        return []
    return list(response.choices[0].message.content.strip().split('\n'))

In [ ]:
system_prompt2 = """
You are a snarky assistant that gets a technical 
topic and a set of weblinks relevant to that topic. 
As a domain expert, you need to analyze those links 
and give a score between 1-10 for each of the weblinks 
based on how relevant they are to the topic. You should 
return a JSON object with the weblink as the key and 
the score as the value.
Example output:
{
    "scores": [
        {"weblink": "https://example.com/link1", "score": 8},
        {"weblink": "https://example.com/link2", "score": 5},
        {"weblink": "https://example.com/link3", "score": 2}
    ]
}
"""

In [ ]:
def get_user_prompt_2(topic, links):
    user_prompt = f"Topic: {topic}\nHere are some weblinks:\n"
    for link in links:
        user_prompt += f"- {link}\n"
    user_prompt += """
    Please score each of the weblinks for relevance to the topic on a 
    scale of 1-10 and return a JSON object with 
    the weblink as the key and the score as the value.
    """
    return user_prompt

In [ ]:
def get_link_scores(topic):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt2},
            {"role": "user", "content": get_user_prompt_2(topic, get_weblinks(topic))}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    parsed = json.loads(result)
    if isinstance(parsed, str):
        parsed = json.loads(parsed)
    return parsed

In [ ]:
def get_topic_explanation(topic):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a concise technical educator. Explain the given topic in 3-4 sentences using markdown formatting."},
            {"role": "user", "content": f"Explain: {topic}"}
        ]
    )
    return response.choices[0].message.content

In [ ]:
def get_top_links(scores, top_k=3):
    score_list = scores["scores"]
    if isinstance(score_list, dict):
        items = sorted(score_list.items(), key=lambda x: x[1], reverse=True)
        return [url for url, _ in items[:top_k]]
    sorted_links = sorted(score_list, key=lambda x: x.get("score", x.get("relevance", 0)), reverse=True)
    return [e.get("weblink", e.get("url", "")) for e in sorted_links[:top_k]]

## Run the pipeline

Change `topic` to any technical subject you're curious about.

In [ ]:
topic = "transformer models in gen AI"

In [ ]:
display(Markdown(get_topic_explanation(topic)))

In [ ]:
scores = get_link_scores(topic)
top_links = get_top_links(scores)

display(Markdown("### Top 3 Links"))
for link in top_links:
    display(Markdown(f"- {link.strip()}"))